<a href="https://colab.research.google.com/github/GODRATAN/FlyRank_ML-Intern/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GODRATAN/FlyRank_ML_Assignment/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### Ranked actions + reason codes

The model ranks content observations by estimated probability of future decline. The ranked queue is intended to help an editor decide which observations deserve human review first.

| Priority signal | Reason code | Suggested action |
|---|---|---|
| High predicted decline risk + meaningful search exposure | `HIGH_DECLINE_RISK` | Review the page for possible content refresh |
| High predicted decline risk + weak CTR relative to position | `HIGH_DECLINE_LOW_CTR` | Review title, snippet, search intent, and content alignment |
| High predicted decline risk + limited current evidence | `LOW_EVIDENCE_RISK` | Monitor and gather more evidence before changing the page |
| Lower predicted decline risk | `MONITOR` | No immediate action; continue monitoring |

The score is a prioritization signal, not a decision to change a page automatically. Reason codes explain why an observation entered the queue and should be checked by a human before action.

In [1]:
# ML-10 — Build ranked action queue

import os
import numpy as np
import pandas as pd

print("Section 1 definitions loaded.")
print("Action categories:")
print([
    "HIGH_DECLINE_RISK",
    "HIGH_DECLINE_LOW_CTR",
    "LOW_EVIDENCE_RISK",
    "MONITOR"
])

Section 1 definitions loaded.
Action categories:
['HIGH_DECLINE_RISK', 'HIGH_DECLINE_LOW_CTR', 'LOW_EVIDENCE_RISK', 'MONITOR']


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

The playbook is intended for content editors or SEO analysts who need to decide which content observations to review first.

The model provides a ranked, decision-support queue. A high score means that the observation is ranked as having a higher measured probability of the future-decline outcome used in this experiment. It does not mean that the page will decline or that a particular edit will improve performance.

### Limits

- The model was evaluated on a specific warehouse slice using a March 2026 feature window and an April 2026 future-decline definition.
- The reported performance comes from a client-grouped holdout, so it measures generalization to the held-out clients in this experiment rather than every possible future client.
- Search and Analytics availability varies across clients and periods, so missing or unavailable data can affect the queue.
- The model does not observe every factor that may affect future content performance, such as changes in search intent, SERP conditions, or business context.
- The model supports prioritization and human review; it is not a production automation system.

In [2]:
# ML-10 — Intended use and limits

intended_users = [
    "content editors",
    "SEO analysts"
]

intended_use = (
    "Rank content observations for human review and prioritization."
)

limits = [
    "Specific March 2026 feature window and April 2026 outcome definition",
    "Evaluated on held-out clients in this experiment",
    "Data availability differs across clients and periods",
    "Does not capture all search-intent, SERP, or business context",
    "Decision-support only; not automatic content changes"
]

print("Intended users:", ", ".join(intended_users))
print("Intended use:", intended_use)

print("\nKey limits:")
for item in limits:
    print("-", item)

Intended users: content editors, SEO analysts
Intended use: Rank content observations for human review and prioritization.

Key limits:
- Specific March 2026 feature window and April 2026 outcome definition
- Evaluated on held-out clients in this experiment
- Data availability differs across clients and periods
- Does not capture all search-intent, SERP, or business context
- Decision-support only; not automatic content changes


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review before action

A human reviewer should check each recommendation before making any content change.

For a high-priority recommendation, the reviewer should check:

1. Whether the page and its current search performance are still relevant.
2. Whether search intent has changed or the observed CTR pattern has a clear explanation.
3. Whether recent content, title, metadata, or site changes already explain the signal.
4. Whether the page has business, legal, brand, or editorial context that is not represented in the model.
5. Whether the proposed action is appropriate after reviewing the actual page.

The model should provide a reason for prioritization, but the human reviewer makes the final action decision.

### No-go list

The following should **not** be automated by this playbook:

- Automatically publishing or rewriting page content.
- Automatically changing titles, metadata, or internal links.
- Automatically deleting, redirecting, or de-indexing pages.
- Automatically making decisions involving legal, regulatory, brand, or sensitive editorial content.
- Automatically treating a high model score as proof that a page needs a specific change.
- Automatically measuring a recommendation as successful without human review and an appropriate evaluation design.

In [3]:
# ML-10 — Human review and no-go rules

human_review_required = True

review_checks = [
    "Check current page and search context",
    "Check search intent and CTR context",
    "Check recent content/site changes",
    "Check business, legal, brand, and editorial context",
    "Confirm the recommended action is appropriate",
]

no_go_automation = [
    "Automatic content publishing or rewriting",
    "Automatic title or metadata changes",
    "Automatic deletion, redirect, or de-indexing",
    "Automatic legal/brand/editorial decisions",
    "Treating model score as proof of required action",
    "Automatic success measurement without human/evaluation review",
]

print("Human review required:", human_review_required)

print("\nRequired review checks:")
for item in review_checks:
    print("-", item)

print("\nNo-go automation:")
for item in no_go_automation:
    print("-", item)

Human review required: True

Required review checks:
- Check current page and search context
- Check search intent and CTR context
- Check recent content/site changes
- Check business, legal, brand, and editorial context
- Confirm the recommended action is appropriate

No-go automation:
- Automatic content publishing or rewriting
- Automatic title or metadata changes
- Automatic deletion, redirect, or de-indexing
- Automatic legal/brand/editorial decisions
- Treating model score as proof of required action
- Automatic success measurement without human/evaluation review


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring and retrain triggers

The playbook should be checked regularly rather than assumed to stay reliable.

**Monitor:**
- Precision@20 and Precision@50 on a recent reviewed sample.
- The proportion of rows with usable GSC and GA4 data.
- Changes in the distributions of the five model features.
- The percentage of recommendations that human reviewers reject.
- Whether the model's high-priority queue is still producing useful review candidates.

**Consider retraining when:**
- Recent precision@20 or precision@50 falls materially below the validated reference result.
- Feature distributions change substantially from the training data.
- Data availability changes enough to alter the composition of the ranked queue.
- Human reviewers frequently reject high-priority recommendations for the same reason.
- The definition of the business outcome or decision changes.

Retraining should be accompanied by a fresh leakage check, an honest validation split, and comparison with the frozen baseline before the updated model is used for decision support.

In [4]:
# ML-10 — Monitoring / retrain triggers

monitoring_signals = {
    "precision_at_20": "Track on a recent reviewed sample",
    "precision_at_50": "Track on a recent reviewed sample",
    "data_availability": "Monitor usable GSC / GA4 coverage",
    "feature_distribution": "Compare recent feature distributions with training data",
    "human_rejection_rate": "Track how often reviewers reject recommendations",
    "queue_usefulness": "Check whether high-priority rows remain useful review candidates",
}

retrain_triggers = {
    "performance_drop": "Recent Precision@20 or Precision@50 falls materially",
    "feature_drift": "Feature distributions change substantially",
    "availability_shift": "Data coverage changes enough to alter the queue",
    "reviewer_rejection": "Repeated reviewer rejection of high-priority recommendations",
    "decision_change": "The underlying business decision or outcome definition changes",
}

print("Monitoring signals:")
for signal, description in monitoring_signals.items():
    print(f"- {signal}: {description}")

print("\nRetrain triggers:")
for trigger, description in retrain_triggers.items():
    print(f"- {trigger}: {description}")

print("\nAny retraining should include:")
print("- fresh leakage audit")
print("- honest validation split")
print("- comparison with the frozen baseline")

Monitoring signals:
- precision_at_20: Track on a recent reviewed sample
- precision_at_50: Track on a recent reviewed sample
- data_availability: Monitor usable GSC / GA4 coverage
- feature_distribution: Compare recent feature distributions with training data
- human_rejection_rate: Track how often reviewers reject recommendations
- queue_usefulness: Check whether high-priority rows remain useful review candidates

Retrain triggers:
- performance_drop: Recent Precision@20 or Precision@50 falls materially
- feature_drift: Feature distributions change substantially
- availability_shift: Data coverage changes enough to alter the queue
- reviewer_rejection: Repeated reviewer rejection of high-priority recommendations
- decision_change: The underlying business decision or outcome definition changes

Any retraining should include:
- fresh leakage audit
- honest validation split
- comparison with the frozen baseline


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Exports for the paper

#### Decay / refresh insight

In this experiment, content decay is represented by the future-decline outcome defined for the April 2026 window. A higher model score means an observation is ranked as having a higher measured probability of that future decline outcome.

These observations can therefore be placed earlier in a refresh-review queue. The model does not show that content age or refreshing a page causes the observed outcome, so refresh remains a human decision-support action.

#### Cost / value thinking

Reviewer time is limited. A practical triage approach is to review higher-risk observations first and use current search exposure as additional context for potential value.

Higher exposure can make a review more consequential because more search activity is associated with the observation, but this is not a monetary ROI estimate. Actual value depends on business context and the outcome of human review.

#### Paper exports

The notebook will regenerate:

- `work/outputs/action_playbook_queue.csv` — ranked action queue for the paper.
- `work/outputs/action_playbook_metrics.json` — validation metrics and run details used as a reproducibility receipt.

The queue CSV stays out of Git by design. The metrics JSON can be committed as a traceable receipt.

In [5]:
# ML-10 — Export ranked action queue and paper metrics

import os
import json
import numpy as np
import pandas as pd

from google.colab import userdata
from datasets import load_dataset

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# --------------------------------------------------
# 1. Load March + April partitions
# --------------------------------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

BASE = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance"
)

needed_cols = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_data_available",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
]


def load_month(month):
    ds = load_dataset(
        "FlyRank/internship-warehouse",
        "fact_content_daily_performance",
        data_files={
            "train": f"{BASE}/month={month}/data_0.parquet"
        },
        token=HF_TOKEN,
    )["train"]

    return ds.select_columns(needed_cols).to_pandas()


# March = feature window
march_raw = load_month("2026-03")
march_raw = march_raw[
    march_raw["gsc_data_available"] == True
].copy()

march_raw["gsc_avg_position"] = (
    march_raw["gsc_avg_position"].replace(0, np.nan)
)

march_agg = (
    march_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        march_impressions=("gsc_impressions", "sum"),
        march_clicks=("gsc_clicks", "sum"),
        march_avg_position=("gsc_avg_position", "mean"),
        march_days_observed=("report_date", "nunique"),
    )
)

del march_raw


# April = future outcome window
april_raw = load_month("2026-04")
april_raw = april_raw[
    april_raw["gsc_data_available"] == True
].copy()

april_agg = (
    april_raw
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        april_impressions=("gsc_impressions", "sum"),
        april_days_observed=("report_date", "nunique"),
    )
)

del april_raw


# --------------------------------------------------
# 2. Build the validated modeling dataset
# --------------------------------------------------

model_data = march_agg.merge(
    april_agg,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_data["march_daily_impressions"] = (
    model_data["march_impressions"]
    / model_data["march_days_observed"]
)

model_data["april_daily_impressions"] = (
    model_data["april_impressions"]
    / model_data["april_days_observed"]
)

eligible = (
    (model_data["march_impressions"] >= 100)
    & (model_data["march_days_observed"] >= 14)
    & (model_data["april_days_observed"] >= 14)
)

model_data = model_data[eligible].copy()

model_data["future_decline_label"] = (
    model_data["april_daily_impressions"]
    <= 0.80 * model_data["march_daily_impressions"]
).astype(int)

model_data["march_ctr"] = (
    model_data["march_clicks"]
    / model_data["march_impressions"].replace(0, np.nan)
)


feature_cols = [
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_days_observed",
]


# --------------------------------------------------
# 3. Client-grouped validation
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_data,
        groups=model_data["client_hash_id"]
    )
)

train_data = model_data.iloc[train_idx].copy()
test_data = model_data.iloc[test_idx].copy()


# --------------------------------------------------
# 4. Train Logistic Regression
# --------------------------------------------------

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(
    train_data[feature_cols],
    train_data["future_decline_label"]
)

test_data["predicted_probability"] = model.predict_proba(
    test_data[feature_cols]
)[:, 1]


# --------------------------------------------------
# 5. Add human-readable action + reason code
# --------------------------------------------------

# Starting triage thresholds for the playbook.
high_risk = test_data["predicted_probability"] >= 0.50
meaningful_exposure = test_data["march_impressions"] >= 100

test_data["reason_code"] = "MONITOR"
test_data["action"] = "MONITOR"

test_data.loc[
    high_risk & meaningful_exposure,
    "reason_code"
] = "HIGH_DECLINE_RISK_VISIBLE"

test_data.loc[
    high_risk & meaningful_exposure,
    "action"
] = "REVIEW_REFRESH"

test_data.loc[
    high_risk & ~meaningful_exposure,
    "reason_code"
] = "HIGH_DECLINE_RISK_LOW_EVIDENCE"

test_data.loc[
    high_risk & ~meaningful_exposure,
    "action"
] = "REVIEW_MORE_DATA"


# --------------------------------------------------
# 6. Rank the queue
# --------------------------------------------------

queue = (
    test_data
    .sort_values(
        "predicted_probability",
        ascending=False
    )
    .reset_index(drop=True)
)

queue.insert(
    0,
    "rank",
    np.arange(1, len(queue) + 1)
)


# --------------------------------------------------
# 7. Export ranked queue
# --------------------------------------------------

os.makedirs("work/outputs", exist_ok=True)

queue_columns = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "predicted_probability",
    "action",
    "reason_code",
    "march_impressions",
    "march_clicks",
    "march_ctr",
    "march_avg_position",
    "march_days_observed",
]

queue_path = "work/outputs/action_playbook_queue.csv"

queue[queue_columns].to_csv(
    queue_path,
    index=False
)


# --------------------------------------------------
# 8. Export metrics receipt
# --------------------------------------------------

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()


test_labels = test_data["future_decline_label"].to_numpy()
model_scores = test_data["predicted_probability"].to_numpy()

metrics = {
    "validation": "client_grouped",
    "random_seed": 42,
    "train_rows": int(len(train_data)),
    "test_rows": int(len(test_data)),
    "train_clients": int(
        train_data["client_hash_id"].nunique()
    ),
    "test_clients": int(
        test_data["client_hash_id"].nunique()
    ),
    "client_overlap": int(
        len(
            set(train_data["client_hash_id"])
            & set(test_data["client_hash_id"])
        )
    ),
    "test_base_rate": float(
        test_data["future_decline_label"].mean()
    ),
    "precision_at_20": float(
        precision_at_k(model_scores, test_labels, 20)
    ),
    "precision_at_50": float(
        precision_at_k(model_scores, test_labels, 50)
    ),
    "feature_cols": feature_cols,
    "queue_rows": int(len(queue)),
}

metrics_path = "work/outputs/action_playbook_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)


print("Queue exported to:", queue_path)
print("Metrics exported to:", metrics_path)

print("\nTop 10 action queue:")
print(
    queue[
        [
            "rank",
            "predicted_probability",
            "action",
            "reason_code",
            "march_impressions",
        ]
    ]
    .head(10)
    .to_string(index=False)
)

print("\nValidation metrics:")
print(json.dumps(metrics, indent=2))

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  134MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

Queue exported to: work/outputs/action_playbook_queue.csv
Metrics exported to: work/outputs/action_playbook_metrics.json

Top 10 action queue:
 rank  predicted_probability         action               reason_code  march_impressions
    1               0.622306 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE             107584
    2               0.605438 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE              82376
    3               0.603823 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE              63201
    4               0.602554 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE             143019
    5               0.594637 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE              58278
    6               0.590333 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE              83788
    7               0.587275 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE              48049
    8               0.586137 REVIEW_REFRESH HIGH_DECLINE_RISK_VISIBLE              38865
    9               0.583892 REVIEW_REFRESH HIGH_DECLINE

## Self-check

Before you submit, confirm each line honestly:

- [YES] Every section above is filled — markdown thinking AND the code that backs it
- [YES] The notebook runs top to bottom with no errors (Runtime → Run all)
- [YES], No client names, URLs, or private queries anywhere
- [YES] My claims use careful words: observed, measured, directional, decision-support
- [YES] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.